# Test the trained 2D and 4D Green's-function models

This notebook loads the saved models from `Models_PDEloss_2D` and `Models_PDEloss_4D`, evaluates the two analytic test problems in `test_functions.py`, and reports the sup norm and relative $L^2$ error. The analytic forcing $f=\Delta u$ is used in the Green's-function quadrature.

In [ ]:
from pathlib import Path
import gc
import pandas as pd
import torch
import torch.nn as nn
from deepsplines.ds_modules import dsnn

from grid import gLLNodesAndWeights
from test_functions import TEST_CASES

torch.set_default_dtype(torch.float64)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
ROOT = Path.cwd()
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(exist_ok=True)
NGRID = 22
CHUNK_SIZE_4D = 32768
print(f'device: {device}')

## Model definitions required by the saved full-model checkpoints

In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, in_dim, mapping_size=64, scale=5.0, mu=5):
        super().__init__()
        generator = torch.Generator(device=device)
        generator.manual_seed(1234)
        if mapping_size % 2 == 0:
            b_pos = torch.randn(in_dim, mapping_size // 2, generator=generator, device=device) * scale + mu
        else:
            b_pos = torch.randn(in_dim, mapping_size // 2 + 1, generator=generator, device=device) * scale + mu
        b_neg = torch.randn(in_dim, mapping_size // 2, generator=generator, device=device) * scale - mu
        self.register_buffer('B', torch.cat([b_pos, b_neg], dim=-1))

    def forward(self, x):
        projection = 2 * torch.pi * x @ self.B
        return torch.cat([torch.sin(projection), torch.cos(projection)], dim=-1)


class NeuralNet_4D(dsnn.DSModule, nn.Module):
    def __init__(self, net_dim, act=nn.ReLU(), use_norm=False, mapping_size=64, scale=5.0, use_fourier=True, mu=5):
        super().__init__()
        self.use_fourier = use_fourier
        self.use_norm = use_norm
        self.act = act
        self.layers = nn.ModuleList()
        self.fc_ds = nn.ModuleList()
        self.norms = nn.ModuleList()
        input_dim = net_dim[0]
        self.ff = FourierFeatures(input_dim, mapping_size, scale, mu).to(device) if use_fourier else None
        if use_fourier:
            input_dim = 2 * mapping_size
        dims = [input_dim] + net_dim[1:]
        self.depth = len(dims) - 1
        for i in range(self.depth - 1):
            self.layers.append(nn.Linear(dims[i], dims[i + 1]))
            if use_norm:
                self.norms.append(nn.LayerNorm(dims[i + 1]))
        self.final = nn.Linear(dims[-2], dims[-1])

    def forward(self, x):
        if self.use_fourier:
            x = self.ff(x)
        for i in range(self.depth - 1):
            x = self.layers[i](x)
            if self.use_norm:
                x = self.norms[i](x)
            x = self.act(x)
        return self.final(x)


class NeuralNet_2D(dsnn.DSModule, nn.Module):
    def __init__(self, net_dim, act=nn.ReLU(), use_norm=False, mapping_size=64, scale=5.0, use_fourier=False, mu=5):
        super().__init__()
        self.use_fourier = use_fourier
        self.use_norm = use_norm
        self.act = act
        self.Rank = net_dim[-1]
        net_dim = net_dim[:-1]
        input_dim = net_dim[0]
        self.ff = FourierFeatures(input_dim, mapping_size, scale, mu) if use_fourier else None
        ff_dim = 2 * mapping_size if use_fourier else input_dim
        dims = [ff_dim] + list(net_dim[1:])
        self.xy_layers = nn.ModuleList()
        self.xy_norms = nn.ModuleList()
        self.st_layers = nn.ModuleList()
        self.st_norms = nn.ModuleList()
        for i in range(len(dims) - 1):
            self.xy_layers.append(nn.Linear(dims[i], dims[i + 1]))
            self.st_layers.append(nn.Linear(dims[i], dims[i + 1]))
            if use_norm:
                self.xy_norms.append(nn.LayerNorm(dims[i + 1]))
                self.st_norms.append(nn.LayerNorm(dims[i + 1]))
        self.xy_final = nn.Linear(dims[-1], self.Rank)
        self.st_final = nn.Linear(dims[-1], self.Rank)

    def forward(self, xy, st):
        if self.use_fourier:
            xy, st = self.ff(xy), self.ff(st)
        for i, layer in enumerate(self.xy_layers):
            xy = layer(xy)
            if self.use_norm:
                xy = self.xy_norms[i](xy)
            xy = self.act(xy)
        for i, layer in enumerate(self.st_layers):
            st = layer(st)
            if self.use_norm:
                st = self.st_norms[i](st)
            st = self.act(st)
        phi = self.xy_final(xy)
        psi = self.st_final(st)
        return torch.einsum('...ijr,...klr->...ijkl', phi, psi)

## Load checkpoints and evaluate each Green's kernel

In [ ]:
paths = {
    'Plan A (2D)': ROOT / 'Models_PDEloss_2D' / 'PINN_model_0' / 'models' / 'model_single.pth',
    'Plan B (4D)': ROOT / 'Models_PDEloss_4D' / 'PINN_model_0' / 'models' / 'model_single.pth',
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f'{name}: {path}')

x, wx = gLLNodesAndWeights(NGRID)
y, wy = gLLNodesAndWeights(NGRID)
X, Y = torch.meshgrid(x, y, indexing='ij')
W = wx[:, None] * wy[None, :]
xy = torch.stack([X, Y], dim=-1).unsqueeze(0).to(device)

def load_full_model(path):
    # Full PyTorch checkpoints use pickle; load only checkpoints you trust.
    model = torch.load(path, map_location=device, weights_only=False)
    return model.to(device).eval()

@torch.inference_mode()
def evaluate_kernel_2d(model):
    return model(xy, xy).squeeze(0)

@torch.inference_mode()
def evaluate_kernel_4d(model, chunk_size=CHUNK_SIZE_4D):
    X4, Y4, S4, T4 = torch.meshgrid(x, y, x, y, indexing='ij')
    coords = torch.stack([X4, Y4, S4, T4], dim=-1).reshape(-1, 4)
    parts = []
    for start in range(0, coords.shape[0], chunk_size):
        parts.append(model(coords[start:start + chunk_size].to(device)).reshape(-1).cpu())
    return torch.cat(parts).reshape(NGRID + 1, NGRID + 1, NGRID + 1, NGRID + 1).to(device)

kernels = {}
model_2d = load_full_model(paths['Plan A (2D)'])
kernels['Plan A (2D)'] = evaluate_kernel_2d(model_2d)
del model_2d
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_4d = load_full_model(paths['Plan B (4D)'])
kernels['Plan B (4D)'] = evaluate_kernel_4d(model_4d)
del model_4d
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print({name: tuple(G.shape) for name, G in kernels.items()})

## Sup norm and relative L2 results

In [ ]:
rows = []
X_device, Y_device, W_device = X.to(device), Y.to(device), W.to(device)

with torch.inference_mode():
    for test_name, (u_exact_fn, forcing_fn) in TEST_CASES.items():
        u_exact = u_exact_fn(X_device, Y_device)
        forcing = forcing_fn(X_device, Y_device)
        weighted_f = W_device * forcing
        for model_name, G in kernels.items():
            u_pred = torch.einsum('ijkl,kl->ij', G, weighted_f)
            error = u_pred - u_exact
            sup_norm = error.abs().max().item()
            relative_l2 = (
                torch.linalg.vector_norm(error)
                / torch.linalg.vector_norm(u_exact)
            ).item()
            rows.append({
                'test': test_name,
                'model': model_name,
                'sup_norm': sup_norm,
                'relative_L2': relative_l2,
            })

results = pd.DataFrame(rows)
results.to_csv(OUTPUT / 'test_results.csv', index=False)
display(results.style.format({'sup_norm': '{:.6e}', 'relative_L2': '{:.6e}'}))

In [ ]:
for row in rows:
    print(
        f"{row['test']:>12s} | {row['model']:<11s} | "
        f"sup norm = {row['sup_norm']:.6e} | relative L2 = {row['relative_L2']:.6e}"
    )